todo
- пересобрать все в функции
- добавить возможность сохранения распределения вероятностей
- впендюрить в основной код

In [ ]:

from pathlib import Path, 
import matplotlib.pyplot as plt
import sys
import re

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT.name != "through_vlm_guidance_research_dev":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from PIL import Image
from pathlib import Path
from modules.utils import choose_device

SOURCE_DIR = Path('../../sources')
device = choose_device()

device = 'mps'


In [ ]:



MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
image = Image.open('../../datasets/image.png').convert('RGB')

img_prompt = 'birch forest'
explanation_max_words = 30


In [ ]:
judge_prompt = f'''
You are an assistant evaluating an image on two independent aspects:
(1) how well it aligns with the meaning of a given text prompt, and
(2) its visual quality.

The text prompt is: "{img_prompt}"

---

PART 1: PROMPT ALIGNMENT (Semantic Fidelity)
Evaluate only the meaning conveyed by the image - ignore visual artifacts.
Focus on:
- Are the correct objects present and depicted in a way that clearly demonstrates their intended roles and actions from the prompt?
- Does the scene illustrate the intended situation or use-case in a concrete and functional way, rather than through symbolic, metaphorical, or hybrid representation?
- If the described usage or interaction is missing or unclear, alignment should be penalized.
- Focus strictly on the presence, roles, and relationships of the described elements - not on rendering quality.

Score from 1 to 5:
5: Fully conveys the prompt's meaning with correct elements
4: Mostly accurate - main elements are correct, with minor conceptual or contextual issues
3: Main subjects are present but important attributes or actions are missing or wrong
2: Some relevant components are present, but key elements or intent are significantly misrepresented
1: Does not reflect the prompt at all

---

PART 2: VISUAL QUALITY (Rendering Fidelity)
Now focus only on how the image looks visually - ignore whether it matches the prompt.
Focus on:
- Are there rendering artifacts, distortions, or broken elements?
- Are complex areas like faces, hands, and shaped objects well-formed and visually coherent?
- Are complex areas like faces, hands, limbs, and object grips well-formed and anatomically correct?
- Is lighting, texture, and perspective consistent across the scene?
- Do elements appear physically coherent - i.e., do objects connect naturally (no floating tools, clipped limbs, or merged shapes)?
- Distortion, warping, or implausible blending of objects should reduce the score.
- Unusual or surreal objects are acceptable if they are clearly rendered and visually deliberate.

Score from 1 to 5:
5: Clean, realistic, and fully coherent - no visible flaws
4: Mostly clean with minor visual issues or stiffness
3: Noticeable visual flaws, but the image is still readable
2: Major visual issues - warped or broken key elements disrupt coherence
1: Severe rendering failure - image appears nonsensical or corrupted

---

Keep each explanation to at most {explanation_max_words} words.

Respond using exactly this format:
### ALIGNMENT SCORE: score
### ALIGNMENT EXPLANATION: explanation
### QUALITY SCORE: score
### QUALITY EXPLANATION: explanation
'''.strip()

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=SOURCE_DIR,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    cache_dir=SOURCE_DIR,
    dtype=torch.bfloat16,
    device_map={'': device}
)

model.eval()

for p in model.parameters():
    p.requires_grad_(False)

In [ ]:

messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': judge_prompt},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors='pt',
)

inputs = inputs.to(device)


Loading weights: 100%|██████████| 625/625 [00:00<00:00, 3322.12it/s]


In [ ]:
with torch.inference_mode():
    output_ids = model.generate(
        **inputs, 
        max_new_tokens=160, 
        do_sample=False
    )



In [ ]:
new_ids = output_ids[:, inputs['input_ids'].shape[1]:]

answer = processor.batch_decode(
    new_ids, 
    skip_special_tokens=True, 
)[0].strip()

print(answer)

In [ ]:
def parse_judge_answer(text: str):
    alignment_score = re.search('### ALIGNMENT SCORE:\s*([1-5])', text)
    alignment_explonation = re.search('### ALIGNMENT EXPLANATION:\s*(.*)', text)
    quality_score = re.search('### QUALITY SCORE:\s*([1-5])', text)
    quality_explonation = re.search('### QUALITY EXPLANATION:\s*(.*)', text)
    
    return {
        'alignment_score': int(alignment_score.group(1)) if alignment_score else None, 
        'alignment_explonation': alignment_explonation.group(1).strip() if alignment_explonation else None, 
        'quality_score': int(quality_score.group(1)) if quality_score else None, 
        'quality_explonation': quality_explonation.group(1).strip() if quality_explonation else None, 
    }


parse_judge_answer(answer)